<a href="https://colab.research.google.com/github/hhy37/-ExcelVBA/blob/master/Copy_of_mnist_horovod_py_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape(60000, 28, 28, 1)
test_images = test_images.reshape(10000, 28, 28, 1)
train_images, test_images = train_images/255, test_images/255
model = tf.keras.Sequential([
 tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
 tf.keras.layers.BatchNormalization(),
 tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
 tf.keras.layers.MaxPooling2D(2,2),
 tf.keras.layers.BatchNormalization(),
 tf.keras.layers.Flatten(),
 tf.keras.layers.Dense(128, activation='relu'),
 tf.keras.layers.BatchNormalization(),
 tf.keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_images, train_labels, batch_size=32, epochs=10, verbose=1, validation_data=(test_images, test_labels))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 195s 103ms/step - accuracy: 0.9713 - loss: 0.0954 - val_accuracy: 0.9846 - val_loss: 0.0463
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 173s 92ms/step - accuracy: 0.9884 - loss: 0.0377 - val_accuracy: 0.9838 - val_loss: 0.0482
Epoch 3/10
1228/1875 ━━━━━━━━━━━━━━━━━━━━ 57s 88ms/step - accuracy: 0.9929 - loss: 0.0239

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir='TB_logDir', histogram_freq=1)
history = model.fit(train_images, train_labels, batch_size=128, epochs=15, verbose=1,
 validation_data=(test_images, test_labels), callbacks=[tensorboard_callback])
tensorboard --logdir=TB_logD

In [ ]:
import tensorflow as tf
import horovod.tensorflow.keras as hvd

# 1. Initialize Horovod
hvd.init()

# 2. Pin GPU to local rank
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
if gpus:
    tf.config.experimental.set_visible_devices(gpus[hvd.local_rank()], 'GPU')

# 3. Load & Preprocess MNIST Data
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
train_images = train_images.reshape(60000, 28, 28, 1).astype('float32') / 255.0
test_images = test_images.reshape(10000, 28, 28, 1).astype('float32') / 255.0

# 4. Create and Shard Datasets
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_images, train_labels))
    .shard(num_shards=hvd.size(), index=hvd.rank())
    .shuffle(10000)
    .batch(128)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    .shard(num_shards=hvd.size(), index=hvd.rank())
    .batch(128)
)

# 5. Build CNN Model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 6. Scale Learning Rate & Wrap Optimizer
scaled_lr = 0.001 * hvd.size()
optimizer = tf.keras.optimizers.Adam(learning_rate=scaled_lr)
optimizer = hvd.DistributedOptimizer(optimizer)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 7. Define Callbacks
callbacks = [
    # Broadcast initial variable states from rank 0 to all other processes
    hvd.callbacks.BroadcastGlobalVariablesCallback(0),
    # Average metrics across workers during training
    hvd.callbacks.MetricAverageCallback(),
]

# Only write TensorBoard logs on worker 0
if hvd.rank() == 0:
    callbacks.append(
        tf.keras.callbacks.TensorBoard(log_dir='TB_logDir', histogram_freq=1)
    )

# 8. Train the Model
model.fit(
    train_ds,
    epochs=15,
    verbose=1 if hvd.rank() == 0 else 0,
    validation_data=test_ds,
    callbacks=callbacks
)